# 02 - Flattening Layer (Phase 2)

Purpose: rebuild deterministic parquet contracts from read-only Mongo base collections.

Inputs: `players`, `bet_transactions`, `deposittransactions`, `withdrawaltransactions`, `bonustransactions`, `useractivitylogs`, `loginlogs`, plus identity/account support collections.

Outputs: parquet files and markdown reports under `data/`.

Core production logic lives in `src/frauddet/flatten.py`; this notebook only orchestrates, previews, and reconciles.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet import config
from frauddet.flatten import OUTPUT_FILES, rebuild_all_flattened_outputs
from frauddet.identity import IdentityMapper, is_objectid_hex, normalize_phone
from frauddet.io_mongo import get_database

pd.set_option("display.max_columns", 80)

## Rebuild

In [2]:
result = rebuild_all_flattened_outputs(config.DATA_DIR)
result.raw_counts, result.parquet_counts, result.report_paths

({'players': 82,
  'bets': 386,
  'deposits': 1075,
  'withdrawals': 79,
  'bonus': 63,
  'activity': 3929,
  'logins': 872},
 {'players': 82,
  'bets': 386,
  'money': 1118,
  'bonus': 63,
  'activity': 3929,
  'logins': 872},
 {'unjoined': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/unjoined_report.md'),
  'reconciliation': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/flatten_reconciliation.md')})

## Schema Previews

In [3]:
frames = {
    name: pd.read_parquet(config.DATA_DIR / filename)
    for name, filename in OUTPUT_FILES.items()
}

for name, frame in frames.items():
    print(f"\n## {name}: {len(frame):,} rows")
    print(frame.dtypes.astype(str).to_string())
    display(frame.head(3))


## players: 82 rows
player_key                         str
phone                              str
created_at         datetime64[us, UTC]
kyc_status                         str
is_deleted                        bool
archived                          bool
referred_by_key                    str
nationality                        str
dob                                str
username_raw                       str


,player_key,phone,created_at,kyc_status,is_deleted,archived,referred_by_key,nationality,dob,username_raw
0,6a0ea9ff174ad3c431d9e16d,757575757,2026-05-21 06:45:19.652000+00:00,Approved,False,False,NaN,Uganda,2000-Jan-01,0757575757
1,6a0eabae174ad3c431d9e6b6,756363636,2026-05-21 06:52:30.870000+00:00,Approved,False,False,NaN,NaN,2000-Jan-01,0756363636
2,6a0eac61174ad3c431d9e947,751234567,2026-05-21 06:55:29.137000+00:00,Approved,False,False,NaN,NaN,1999-Dec-01,0751234567



## bets: 386 rows
player_key                          str
unjoined_class                      str
ticket_id                           str
game_type                           str
status                              str
result                              str
stake                             int64
stake_real                        int64
stake_bonus                       int64
is_free_bet                        bool
payout                          float64
potential_payout                float64
total_odds                      float64
currency                            str
created_at          datetime64[us, UTC]
settled_at_proxy    datetime64[us, UTC]
bet_type                            str
n_selections                      int64
min_part_odds                   float64
max_part_odds                   float64
sports                           object


,player_key,unjoined_class,ticket_id,game_type,status,result,stake,stake_real,stake_bonus,is_free_bet,payout,potential_payout,total_odds,currency,created_at,settled_at_proxy,bet_type,n_selections,min_part_odds,max_part_odds,sports
0,6a215a2a7364bea0f4f5a90f,NaN,96910293-b16d-4641-b0dc-0e4e3300babd,Sports-book,SETTLED,WIN,1000,1000,0,False,1317.5,1550.0,1.55,INR,2026-05-26 05:32:50.833000+00:00,2026-05-26 20:26:20.812000+00:00,single,1,1.55,1.55,[football]
1,6a215a2a7364bea0f4f5a90f,NaN,c7e8283b-c509-43c1-8672-c1a216f93fb8,Sports-book,SETTLED,WIN,1000,1000,0,False,1258.0,1480.0,1.48,INR,2026-05-26 09:05:50.430000+00:00,2026-05-26 20:26:20.738000+00:00,single,1,1.48,1.48,[football]
2,6a0ed517833df31e9ad37961,NaN,6cd3fc1e-3954-41c7-b2fa-3731e02207e8,Sports-book,SETTLED,LOSE,1000,1000,0,False,0.0,2450.0,2.45,INR,2026-05-26 20:48:43.918000+00:00,2026-05-26 23:56:26.544000+00:00,single,1,2.45,2.45,[football]



## money: 1,118 rows
player_key                                  str
unjoined_class                              str
txn_type                                    str
transaction_id                              str
amount                                    int64
currency                                    str
payment_method                              str
account_number                              str
final_status                                str
requested_at                datetime64[us, UTC]
finalized_at                datetime64[us, UTC]
execution_type                              str
recipient_normalized                        str
is_third_party_recipient                   bool
bonus_tags                               object
is_money_in                                bool
is_money_out                               bool
is_pending_withdrawal                      bool


...

,player_key,unjoined_class,source_id,txn_type,amount,bonus_type,ref_trans_id,ref_kind,currency,created_at
0,6a215a2a7364bea0f4f5a90f,NaN,6a14589ab76d79e0209cf70e,ALLOCATED,800.0,First Deposit,TXNY_MPL7CMMP_7069B12A63A6,deposit,UGX,2026-05-25 14:11:38.818000+00:00
1,6a0eac61174ad3c431d9e947,NaN,6a1549beb76d79e020b515d8,ALLOCATED,800.0,First Deposit,TXNY_MPM9JG1A_8EC8872C1F32,deposit,UGX,2026-05-26 07:20:30.986000+00:00
2,6a215a2a7364bea0f4f5a90f,NaN,6a1601edb76d79e020bbe445,RELEASE,200.0,First Deposit,96910293-b16d-4641-b0dc-0e4e3300babd,bet,UGX,2026-05-26 20:26:21.379000+00:00



## activity: 3,929 rows
player_key                        str
unjoined_class                    str
source_id                         str
action                            str
client_ip                         str
page                              str
device_type                       str
created_at        datetime64[us, UTC]


,player_key,unjoined_class,source_id,action,client_ip,page,device_type,created_at
0,NaN,unknown,6a0ea89609b73d64e79ce37a,LOGOUT,80.227.165.206,/,desktop,2026-05-21 06:39:18.872000+00:00
1,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa09174ad3c431d9e185,LOGIN,80.227.165.206,/,desktop,2026-05-21 06:45:29.857000+00:00
2,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa3b174ad3c431d9e434,NAVIGATE,80.227.165.206,/dashboard/deposit,desktop,2026-05-21 06:46:19.872000+00:00



## logins: 872 rows
player_key                        str
unjoined_class                    str
source_id                         str
fingerprint                       str
user_type                         str
success                        object
failure_reason                 object
created_at        datetime64[us, UTC]


,player_key,unjoined_class,source_id,fingerprint,user_type,success,failure_reason,created_at
0,NaN,staff,6a0ea980174ad3c431d9de56,10c1df6db2524a9848b00cf27110cab3510e8d466d42a1...,MANAGER,True,None,2026-05-21 06:43:12.403000+00:00
1,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa09174ad3c431d9e183,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d26045...,PLAYER,True,None,2026-05-21 06:45:29.105000+00:00
2,NaN,staff,6a0eaaa0174ad3c431d9e4c3,10c1df6db2524a9848b00cf27110cab3510e8d466d42a1...,MANAGER,True,None,2026-05-21 06:48:00.564000+00:00


## Row Counts

In [4]:
pd.DataFrame(
    {
        "raw_mongo_rows": pd.Series(result.raw_counts),
        "parquet_rows": pd.Series(result.parquet_counts),
    }
)

,raw_mongo_rows,parquet_rows
activity,3929.0,3929.0
bets,386.0,386.0
bonus,63.0,63.0
deposits,1075.0,NaN
logins,872.0,872.0
money,NaN,1118.0
players,82.0,82.0
withdrawals,79.0,NaN


## Player Join Rates

In [5]:
join_rows = []
for name, frame in frames.items():
    if "player_key" not in frame.columns:
        continue
    nulls = int(frame["player_key"].isna().sum())
    join_rows.append(
        {
            "table": name,
            "rows": len(frame),
            "null_player_key": nulls,
            "join_rate": None if len(frame) == 0 else 1 - nulls / len(frame),
        }
    )

pd.DataFrame(join_rows).sort_values("table")

,table,rows,null_player_key,join_rate
4,activity,3929,254,0.935353
1,bets,386,16,0.958549
3,bonus,63,7,0.888889
5,logins,872,440,0.495413
2,money,1118,3,0.997317
0,players,82,0,1.000000


## Hardening Checks

In [6]:
print("collapsed_withdrawal_count", result.collapsed_withdrawal_count)
print("anomalies", result.anomalies)
print("identity_collisions", result.identity_collisions)

collapsed_withdrawal_count 43
anomalies []
identity_collisions []


## referredBy Verification

In [7]:
with get_database() as db:
    player_docs = list(
        db[config.get_collection_names()["players"]].find(
            {},
            {
                "_id": 1,
                "username": 1,
                "contactNo": 1,
                "playerId": 1,
                "referralCode": 1,
                "referredBy": 1,
                "referredCode": 1,
                "isDeleted": 1,
                "createdAt": 1,
            },
        )
    )

player_ids = {str(doc.get("playerId")) for doc in player_docs if doc.get("playerId") is not None}
referral_codes = {str(doc.get("referralCode")) for doc in player_docs if doc.get("referralCode")}
player_keys = {str(doc.get("_id")) for doc in player_docs if doc.get("_id") is not None}
phones = {
    phone
    for doc in player_docs
    for phone in (normalize_phone(doc.get("username")), normalize_phone(doc.get("contactNo")))
    if phone
}
mapper = IdentityMapper.from_players(player_docs)

def referred_by_format(value):
    text = str(value).strip()
    if text in player_ids:
        return "playerId"
    if text in referral_codes:
        return "referralCode"
    if text in player_keys or is_objectid_hex(text):
        return "players._id_hex"
    if normalize_phone(text) in phones:
        return "phone"
    return "unknown"

referred_by_rows = []
for doc in player_docs:
    value = doc.get("referredBy")
    if value is None:
        continue
    resolved = mapper.resolve(value)
    referred_by_rows.append(
        {
            "player_key": str(doc.get("_id")),
            "username": doc.get("username"),
            "referredBy": value,
            "referredCode": doc.get("referredCode"),
            "format": referred_by_format(value),
            "resolved_key": resolved.player_key,
        }
    )

referred_by_df = pd.DataFrame(referred_by_rows)
display(referred_by_df)

,player_key,username,referredBy,referredCode,format,resolved_key
0,6a0f0df7a5d7ce78c657a62f,0751212121,10003905,Y6YCDR,playerId,6a0ef9cca5d7ce78c65765d9
1,6a14351b6ac0a3933202239e,0757676767,10003905,Y6YCDR,playerId,6a0ef9cca5d7ce78c65765d9


## Reconciliation

In [8]:
print((config.DATA_DIR / "flatten_reconciliation.md").read_text(encoding="utf-8"))

# Flatten Reconciliation

| Source | Raw Mongo rows | Parquet rows | Delta explanation |
| --- | ---: | ---: | --- |
| players | 82 | 82 | One row per player; no drops. |
| bets | 386 | 386 | One row per ticket; unjoined rows kept. |
| deposits | 1075 | included in money | No deposit deduplication; all statuses kept and flags determine money-in. |
| withdrawals | 79 | 43 collapsed groups | Lifecycle collapse by `transactionId`; status order completed > declined > failed > pending > initiated. |
| money | 1075 deposits + 43 collapsed withdrawals | 1118 | ASSERTED: money rows equal deposits plus collapsed withdrawals. |
| bonus | 63 | 63 | One row per bonus transaction; currency is assumed `UGX` because source has no currency field. |
| activity | 3929 | 3929 | One row per activity event; unjoined rows kept. |
| logins | 872 | 872 | One row per login log; staff rows kept with null player_key. |

## Withdrawal Account Anomaly

Status x `toAccountId` resolution:

| status | cashaccounts | 

Expected deltas explained above:

- Withdrawals collapse from lifecycle documents to one row per `transactionId`.
- Deposits are not deduplicated; all statuses are kept and interpreted through boolean flags.
- Unjoined source rows are kept with `player_key = NULL` and `unjoined_class`.
- Bonus currency is assumed from `config.yaml` because `bonustransactions` has no currency field.

There are zero unexplained drops in the current run.

## Reports

In [9]:
for name, path in result.report_paths.items():
    print(f"\n# {name}: {path}\n")
    print(path.read_text(encoding="utf-8"))


# unjoined: C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book\data\unjoined_report.md

# Unjoined Report

Rows are kept in parquet with `player_key = NULL` and excluded later at feature time.

Known orphan-wallet IDs called out from Phase 1b:
- `0751231231`
- `0754321012`

## Identity Collisions

No normalized phone collisions detected.

## bets

| unjoined_class | count | sample IDs |
| --- | ---: | --- |
| pre_registration | 16 | 17759968-d60a-4a98-b73a-f99923f8f4aa, 797ab76e-19bb-4648-9d46-46eac29c7a5e, ca52d46a-0d45-4bdf-b894-b3cfc5184b34, 7a7c39c4-82eb-4d06-ba96-bfbb66b9d0f3, 40fd0c8c-e00c-490c-a0fd-aa465ab9784f, 870cf27a-dbf8-4660-adc9-853881d402bb, 5efc8ac3-2d21-44b0-af8a-a814680c4090, d9a2bf87-c468-40b7-b1fb-88338440220d, 00e6b9d7-98c6-49be-b11a-efa18ff2e001, 2e481ba3-a524-4bb1-a371-12d774658f6a |

## money

| unjoined_class | count | sample IDs |
| --- | ---: | --- |
| pre_registration | 3 | TXNY_MPTJ0I47_D08E1808A766, TXNY_MPWC0JL4_8FF41E8C550

## FINDINGS — Notebook 02: flattening layer — 2026-06-12
- Verified: Executed hardening run produced `players=82`, `bets=386`, `money=1,118`, `bonus=63`, `activity=3,929`, `logins=872` parquet rows from current Mongo. Money rows reconcile as 1,075 deposits plus 43 collapsed withdrawal transaction groups from 79 lifecycle docs.
- Verified: Reconciliation now asserts row math from an independently computed collapsed-withdrawal count. Coded withdrawal invariants are VERIFIED and `result.anomalies=[]`: wallet-pointing `toAccountId` statuses are only failed/declined; non-reversal missing `fromAccountId` rows are initiated.
- Verified: Staff login rows are no longer buried under unknowns: logins unjoined classes are `staff=407`, `pre_registration=33`, genuine login `unknown=0`. Other unjoined counts remain visible: bets 16, money 3, bonus 7, activity 254 (`pre_registration=248`, `test_pattern=3`, `unknown=3`).
- Verified: No normalized phone collisions were detected in current players (`identity_collisions=[]`), but the mapper now records collisions and resolves reused phones by non-deleted player, then latest `createdAt`.
- Verified: `players.referredBy` has two non-null values in current data; both are `playerId` references (`10003905`) and resolve to `6a0ef9cca5d7ce78c65765d9`. Referral-code exact matching is supported defensively but not used by current rows.
- Surprises: Current live counts differ again from earlier Phase 1/Phase 2 notes because the dev Mongo data changed while hardening was underway, especially deposits (`1,075`).
- Gaps: Bonus transactions still have no currency field; `currency=UGX` is an assumed default from `config.yaml`. Bonus `ref_kind` now distinguishes unresolved references (current count 2) from missing references (current count 0).
- Decisions needed: Operations still needs to confirm whether `deposittransactions.status == "manual_reconciliation"` credits the player; backend/ops should confirm whether pre-registration identities can transact in production; backend should confirm whether `loginlogs.operationType == "Login"` implies success when explicit success fields are absent.
- Updated assumptions: Later phases should read only these parquet contracts, filter money using canonical flags, treat staff login rows separately from unresolved players, and avoid money-vs-bet currency ratios because money is UGX while bets are INR.